# Use Model for Predictions


In [ ]:
from types import SimpleNamespace
import os
from IPython import get_ipython
import numpy as np
import torch
import lightning as pl

from data_module import NPZTensorDataModule
from model import MultiLayerLeakyReLUModel

detect if runnig as notebook or script


In [ ]:
if get_ipython():
    is_running_in_notebook = True
    print("executing as notebook ...")
else:
    is_running_in_notebook = False
    print("executing as script ...")

## Args

### Demo

In [ ]:
# args = SimpleNamespace(
#     # io
#     data_path="data/demo",
#     logs_path="data/logs",
#     predicitions_path="data/predicted/demo",
#     experiment_name="demo_experiment",
#     version="v1",
#     # data
#     batch_size=64,
#     num_workers=4,
#     persistent_workers=True,
#     pin_memory=True,
# )

### laser-pulse-shaping-astra-sim

In [ ]:
args = SimpleNamespace(
    # io
    data_path="data/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    logs_path="data/logs",
    experiment_name="laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    predicitions_path="data/predicted/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
    version="2025-12-13-13-34-11-ANimeJpN",
    # data
    batch_size=64,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
)

In [ ]:
version_path = os.path.join(args.logs_path, args.experiment_name, args.version)

In [ ]:
prediction_path = os.path.join(version_path, "predict")
os.makedirs(prediction_path, exist_ok=True)

In [ ]:
ckpt_path = os.path.join(version_path, "best.ckpt")

## load data


In [ ]:
data = NPZTensorDataModule(
    path=args.data_path,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    persistent_workers=args.persistent_workers,
    pin_memory=args.pin_memory,
)
data.setup("predict")

## load model


In [ ]:
model = MultiLayerLeakyReLUModel.load_from_checkpoint(ckpt_path)

## Evaluate Best Model


### validation dataset


In [ ]:
trainer = pl.Trainer(logger=False)

## predict


In [ ]:
predictions = trainer.predict(model, datamodule=data)
predictions = torch.concat(predictions, dim=0)

### save


In [ ]:
out_path = os.path.join(args.predicitions_path, args.version, "predictions.npz")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
np.savez(out_path, y=predictions.numpy())